In [1]:
# Carga las librerías, define las rutas y lee el dataset limpio para prepararlo antes de entrenar los modelos.
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
from sklearn.model_selection import train_test_split

# Rutas del proyecto (Datos es una carpeta nueva para guardar los datos ya preparados)
PROYECTO = Path("..")
RESULTADOS = PROYECTO / "Resultados"
DATOS = PROYECTO / "Datos"
DATOS.mkdir(exist_ok=True)

# Orden fijo de las emociones: cada una recibe un número de 0 a 6 para los modelos
EMOCIONES = ["Felicidad", "Neutral", "Tristeza", "Sorpresa", "Asco", "Enojo", "Miedo"]

df = pd.read_csv(RESULTADOS / "dataset_limpio.csv")
print(f"Imágenes cargadas: {len(df)}")
print(df["split"].value_counts())

Imágenes cargadas: 15326
split
train    12261
test      3065
Name: count, dtype: int64


In [2]:
# Lee cada imagen como arreglo de números y crea las etiquetas numéricas, para poder entrenar los modelos.
X = np.zeros((len(df), 100, 100, 3), dtype="uint8")
for i, ruta in enumerate(df["ruta"]):
    X[i] = cv2.cvtColor(cv2.imread(ruta), cv2.COLOR_BGR2RGB)

y = df["emocion"].map({emocion: numero for numero, emocion in enumerate(EMOCIONES)}).to_numpy()

print("Forma de X:", X.shape)
print("Tipo de dato:", X.dtype, "| tamaño en memoria:", round(X.nbytes / 1e6), "MB")
print("Valores de los píxeles:", X.min(), "a", X.max())
print("Etiquetas (0 a 6):", np.unique(y), "| cantidad por emoción:", np.bincount(y))

Forma de X: (15326, 100, 100, 3)
Tipo de dato: uint8 | tamaño en memoria: 460 MB
Valores de los píxeles: 0 a 255
Etiquetas (0 a 6): [0 1 2 3 4 5 6] | cantidad por emoción: [5955 3198 2459 1618  877  864  355]


In [3]:
# Separa los datos en entrenamiento, validación y prueba, manteniendo la proporción de cada emoción.
PROPORCION_VALIDACION = 0.15

indices_train = np.where(df["split"] == "train")[0]
indices_prueba = np.where(df["split"] == "test")[0]

indices_entrenamiento, indices_validacion = train_test_split(
    indices_train, test_size=PROPORCION_VALIDACION, stratify=y[indices_train], random_state=42
)

for nombre, indices in [("Entrenamiento", indices_entrenamiento), ("Validación", indices_validacion), ("Prueba", indices_prueba)]:
    print(f"{nombre}: {len(indices)} imágenes | por emoción: {np.bincount(y[indices], minlength=7)}")

Entrenamiento: 10421 imágenes | por emoción: [4054 2140 1685 1096  609  598  239]
Validación: 1840 imágenes | por emoción: [716 378 297 193 108 106  42]
Prueba: 3065 imágenes | por emoción: [1185  680  477  329  160  160   74]


In [4]:
# Calcula un peso para cada emoción: las que tienen pocas imágenes pesan más, para que el modelo no las ignore.
y_entrenamiento = y[indices_entrenamiento]
cantidades = np.bincount(y_entrenamiento, minlength=7)
pesos = len(y_entrenamiento) / (7 * cantidades)

for emocion, cantidad, peso in zip(EMOCIONES, cantidades, pesos):
    print(f"{emocion:10}: {cantidad:5} imágenes | peso: {peso:.2f}")

Felicidad :  4054 imágenes | peso: 0.37
Neutral   :  2140 imágenes | peso: 0.70
Tristeza  :  1685 imágenes | peso: 0.88
Sorpresa  :  1096 imágenes | peso: 1.36
Asco      :   609 imágenes | peso: 2.44
Enojo     :   598 imágenes | peso: 2.49
Miedo     :   239 imágenes | peso: 6.23


In [5]:
# Guarda los datos ya separados y los pesos en la carpeta Datos
np.save(DATOS / "X_entrenamiento.npy", X[indices_entrenamiento])
np.save(DATOS / "y_entrenamiento.npy", y[indices_entrenamiento])
np.save(DATOS / "X_validacion.npy", X[indices_validacion])
np.save(DATOS / "y_validacion.npy", y[indices_validacion])
np.save(DATOS / "X_prueba.npy", X[indices_prueba])
np.save(DATOS / "y_prueba.npy", y[indices_prueba])
np.save(DATOS / "pesos_clase.npy", pesos)

# Guarda la lista de imágenes de prueba con sus medidas de calidad,
#  para cruzarla después con las predicciones.
prueba = df.iloc[indices_prueba][["archivo", "emocion", "brillo", "contraste", "nitidez", "diferencia_color", "es_gris"]].copy()
prueba["etiqueta"] = y[indices_prueba]
prueba.to_csv(RESULTADOS / "prueba_imagenes.csv", index=False, encoding="utf-8-sig")

for archivo in sorted(DATOS.glob("*.npy")):
    print(f"{archivo.name:25} {archivo.stat().st_size / 1e6:8.1f} MB")
print(f"prueba_imagenes.csv: {len(prueba)} imágenes")

pesos_clase.npy                0.0 MB
X_entrenamiento.npy          312.6 MB
X_prueba.npy                  92.0 MB
X_validacion.npy              55.2 MB
y_entrenamiento.npy            0.1 MB
y_prueba.npy                   0.0 MB
y_validacion.npy               0.0 MB
prueba_imagenes.csv: 3065 imágenes
